In [1]:
from utils.analysis.valuation import BuySellSignalsAnalyzer, SignalsReporter
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE, TRADING_SIGNALS_CONFIG

In [2]:
# 📊 CONFIGURACIÓN
# Las fechas vienen de config.py (TRADING_SIGNALS_CONFIG)
# Personaliza aquí solo si necesitas valores diferentes

# Tickers a analizar
TICKERS = ["INCY"]  # Agrega más: ["AAPL", "MSFT", "GOOGL", ...]

# Lookback para análisis de precios
LOOKBACK_YEARS = 5  # Años de histórico

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
CUSTOM_START = ""  # Ej: "2022-01-01"
CUSTOM_END = ""    # Ej: "2024-12-31"

print("📋 Configuración:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Lookback: {LOOKBACK_YEARS} años")
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    print(f"   Fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    print(f"   Fechas desde config.py: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE}")

📋 Configuración:
   Tickers: 1
   Lookback: 5 años
   Fechas desde config.py: 2020-01-01 → 2025-12-31


In [3]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager
data_manager = DataManager()

# Crear analyzer con configuración
# Si USE_CUSTOM_DATES es True, usar fechas personalizadas
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        start_date=CUSTOM_START,
        end_date=CUSTOM_END,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con configuración de config.py")

reporter = SignalsReporter()
print("✅ Reporter inicializado")

✅ Analyzer con configuración de config.py
✅ Reporter inicializado


In [4]:
# 🔍 ANÁLISIS DE SEÑALES
signals = []

print(f"\n🔍 Analizando {len(TICKERS)} ticker(s)...\n")

for ticker in TICKERS:
    try:
        # El analyzer ya tiene las fechas configuradas
        signal = analyzer.analyze_stock(ticker)
        signals.append(signal)
        
        # Mostrar resumen rápido
        emoji = "🟢" if signal.signal == "COMPRA" else "🔴" if signal.signal == "VENTA" else "🟡"
        print(f"{emoji} {ticker}: {signal.signal} "
              f"(Conf: {signal.confidence:.1f}%, Pot: {signal.upside_potential:+.1f}%)")
        
    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)[:60]}")

print(f"\n📊 Total analizado: {len(signals)}/{len(TICKERS)} tickers")


🔍 Analizando 1 ticker(s)...

Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed

🟢 INCY: COMPRA (Conf: 62.9%, Pot: -0.8%)

📊 Total analizado: 1/1 tickers


In [5]:
# 📊 RESUMEN EN TABLA
if signals:
    summary_df = reporter.to_dataframe(signals)
    display(summary_df)
    
    print()
    reporter.print_summary(signals)
else:
    print("❌ No hay señales para mostrar")

,Ticker,Señal,Confianza,Valoración,Fundamental,Precio Actual,Precio Objetivo,Potencial
0,INCY,COMPRA,62.9%,52.4,90.8,$101,$100,-0.8%




RESUMEN DE SEÑALES

🟢 COMPRAS: 1
   INCY: 62.9% confianza, -0.8% potencial

🔴 VENTAS: 0

🟡 MANTENER: 0


In [6]:
# 🏆 TOP OPORTUNIDADES DE COMPRA
if signals:
    reporter.print_top_opportunities(signals, top_n=5)
else:
    print("❌ No hay señales para analizar")


TOP 5 OPORTUNIDADES DE COMPRA

1. INCY
   Confianza: 62.9%
   Potencial: -0.8%
   Precio: $101 → $100
   Valoración: 52.4 | Fundamental: 90.8
   Razones: ⭐ Empresa de calidad excepcional, 📈 Excelente rentabilidad


In [7]:
# 🟢 DETALLE DE SEÑALES DE COMPRA
compras = [s for s in signals if s.signal == "COMPRA"]

if compras:
    # Ordenar por confianza
    top_compras = sorted(compras, key=lambda x: x.confidence, reverse=True)
    
    print(f"📈 Encontradas {len(compras)} señales de COMPRA\n")
    print("=" * 65)
    
    for signal in top_compras:
        reporter.print_signal(signal)
        print()
else:
    print("ℹ️  No hay señales de COMPRA en este momento")

📈 Encontradas 1 señales de COMPRA

SEÑAL DE INVERSIÓN: INCY

🟢 COMPRA (Confianza: 62.9%)
📊 SCORES:
   Valoración:    🟠 [██████████░░░░░░░░░░] 52.4
   Fundamental:   🟢 [██████████████████░░] 90.8
💰 PRECIOS:
   Actual:        $101
   Objetivo:      $100
   Potencial:     -0.8%

💡 RAZONES:
   ⭐ Empresa de calidad excepcional
   📈 Excelente rentabilidad
   🛡️ Balance sólido - Baja deuda



In [8]:
# 🏭 ANÁLISIS POR SECTOR
# Analiza múltiples tickers de un sector específico

TECH_SECTOR = ["AAPL", "MSFT", "GOOGL", "META", "NVDA", "AMD", "INTC", "CRM"]

print("🔍 Analizando sector tecnológico...\n")
tech_signals = []

for ticker in TECH_SECTOR:
    try:
        signal = analyzer.analyze_stock(ticker)
        tech_signals.append(signal)
    except Exception as e:
        print(f"⚠️  {ticker}: {str(e)[:40]}")

if tech_signals:
    # Mostrar solo las mejores oportunidades
    compras_tech = [s for s in tech_signals if s.signal == "COMPRA"]
    
    if compras_tech:
        print(f"\n✅ {len(compras_tech)} oportunidades de compra en Tech:\n")
        
        tech_df = reporter.to_dataframe(compras_tech)
        display(tech_df.sort_values('Confianza', ascending=False))
    else:
        print("\nℹ️  No hay señales de compra en el sector tech actualmente")

🔍 Analizando sector tecnológico...

Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


Período: 2020-01-01 → 2025-12-24


...

Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


ℹ️  No hay señales de compra en el sector tech actualmente
